# <center> Funcomics from DEA

## <center> Description

- env: squidpy_v06

## <center> Setup

#### **Python**

In [1]:
# Load libraries
import numpy as np
import pandas as pd
import decoupler as dc

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
import sys
from pathlib import Path
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plots
from math import ceil
import matplotlib.pyplot as plt
import plotnine as p9

#### **Paths**

In [ ]:
if 'snakemake' in locals():
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    
    # input
    ## DEA results (above)
    in_deg = ROOT / "output/DEA/objects/calibrate_pvalues/deg_calibrated.tsv"
    ## funcomics
    path_input_objects = ROOT / "output/preprocessing/objects/getGenesets"
    in_collectri=path_input_objects / 'collectri.tsv'
    in_progeny=path_input_objects / 'progeny.tsv'
    in_go_cc=path_input_objects / 'go_cc.tsv'
    in_go_mf=path_input_objects / 'go_mf.tsv'
    in_go_bp=path_input_objects / 'go_bp.tsv'
    in_hallmark=path_input_objects / 'hallmark.tsv'
    in_reactome=path_input_objects / 'reactome.tsv'
    in_KEGG=path_input_objects / 'kegg.tsv'

    # output
    ## plots
    out_plots = ROOT / "output/DEA/plots/funcomics"
    out_plots.mkdir(parents=True, exist_ok=True)
    ## objects
    out_dea_enrich=ROOT / "output/DEA/objects/funcomics/funcomics.tsv"
    out_dea_enrich.parent.mkdir(parents=True, exist_ok=True) # make sure the output folder exists

In [3]:
colname_comparison='contrast'
colname_features='genesymbol'
colname_stat='stat'

## <center> Load data

#### **degs**

In [4]:
deg=pd.read_csv(in_deg,sep='\t',index_col=0)

deg.head()

,gene_id,genesymbol,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,contrast,significant_pvalue_0.05,significant_padj_0.05,direction
1,ENSG00000283293,RN7SK,455.747883,-4.375801,0.553297,-7.908591,2.220446e-16,2.171152e-12,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
2,ENSG00000202198,7SK,303.688625,-3.769521,0.527742,-7.142735,2.220446e-16,2.171152e-12,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
3,ENSG00000202058,RN7SKP80,14.951145,-5.132970,0.858145,-5.981471,3.161915e-13,2.061147e-09,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
4,ENSG00000269900,NaN,43.450875,-2.420166,0.448516,-5.395944,4.900302e-11,2.395758e-07,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos
5,ENSG00000081237,PTPRC,27.428664,-5.929187,1.121367,-5.287462,1.180884e-10,4.618675e-07,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,True,True,higher_HR_control__IFNG_pos


## <center> Funcomics

#### **Load gene sets**

In [5]:
# Load gene sets

# collectri TFs
df_collectri=pd.read_csv(in_collectri, sep='\t')
# progeny
df_progeny=pd.read_csv(in_progeny, sep='\t')
# go_bp
df_go_bp=pd.read_csv(in_go_bp, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# go_mf
df_go_mf=pd.read_csv(in_go_mf, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# go_cc
df_go_cc=pd.read_csv(in_go_cc, sep='\t').rename(columns={'geneset':'source', 'genesymbol':'target'})
# hallmark
df_hallmark=(
    pd.read_csv(in_hallmark, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('HALLMARK','HK'))
)
# KEGG
df_kegg=(
    pd.read_csv(in_KEGG, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('KEGG','KG'))
)
# reactome
df_reactome=(
    pd.read_csv(in_reactome, sep='\t').
    rename(columns={'geneset':'source', 'genesymbol':'target'}).
    assign(source=lambda x:x['source'].str.replace('REACTOME','RE'))
)

## From DEG results

#### **Prepare data**

In [6]:
# prepare input data

mat=deg.pivot_table(index=colname_comparison, columns=colname_features, values=colname_stat)

mat.head()

genesymbol,5_8S_rRNA,7SK,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
contrast,,,,,,,,,,,,,,,,,,,,,
HR_case__IFNG_pos_vs_HR_control__IFNG_pos,3.300121,-7.142735,0.527675,-0.106598,-1.061463,1.527357,-2.611118,0.07572,0.603045,2.032649,...,0.46103,0.101135,0.506536,1.313676,1.029441,-0.327883,0.632131,0.561252,-0.260518,-0.30176


#### **Perform enrichment**

In [7]:
# Perform enrichment with ulm
## only for weighted gene sets

res_lst=[]
input_lst=[
    ('collectri',df_collectri),('progeny',df_progeny),
    ]
for pk,df_input in input_lst:
    ## get results
    res_acts, res_p = dc.mt.ulm(data=mat,net=df_input,verbose=False)
    ## merge act/pvalue results
    res=(
        res_acts.reset_index(names='comparison').
        melt(id_vars='comparison', var_name='source', value_name='act').
        merge(
            res_p.reset_index(names='comparison').melt(id_vars='comparison', var_name='source', value_name='padj')).
        assign(gene_set=pk,method='ulm',direction=None,n_foreground=None,n_background=None).
        sort_values(by='padj', ascending=True))
    ## save results
    res_lst.append(res)
# concat
df_enrich=pd.concat(res_lst)

df_enrich.head()

,comparison,source,act,padj,gene_set,method,direction,n_foreground,n_background
574,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,SPI1,-8.541232,1.021975e-14,collectri,ulm,None,None,None
123,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,ETS1,-6.066729,4.747640e-07,collectri,ulm,None,None,None
519,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,RUNX1,-5.904523,8.557855e-07,collectri,ulm,None,None,None
378,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,NFATC1,-4.235497,4.085288e-03,collectri,ulm,None,None,None
265,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,IRF8,-3.671292,3.451359e-02,collectri,ulm,None,None,None


In [8]:
## Test overlap with query_set enrichment for unweighted gene sets 
## test for significantly up-regulated DEGs


deg2 = deg.copy()
## keep valid DEA rows
deg2 = deg2.dropna(subset=[colname_comparison, colname_features, "padj", "stat"])
## make sure p-values/statistics are numeric
deg2["padj"] = pd.to_numeric(deg2["padj"], errors="coerce")
deg2["stat"] = pd.to_numeric(deg2["stat"], errors="coerce")
deg2 = deg2.dropna(subset=["padj", "stat"])

## run Fisher's
input_lst = [
    ("go_bp", df_go_bp),
    ("go_mf", df_go_mf),
    ("go_cc", df_go_cc),
    ("hallmark", df_hallmark),
    ("kegg", df_kegg),
    ("reactome", df_reactome),
]
res_lst = []
for comparison, df_cmp in deg2.groupby(colname_comparison):
    ## background: all genes tested in this DEA contrast
    background_genes = set(df_cmp[colname_features].dropna().unique())
    n_bg = len(background_genes)
    ## foreground (e.g., significantly upregulated genes)
    foreground = set( df_cmp.loc[(df_cmp["padj"] < 0.05) & (df_cmp["stat"] > 0),colname_features,].dropna().unique() )
    ## skip if no foreground 
    if len(foreground) == 0:
        continue

    for pk, df_input in input_lst:
        ## run Fisher's
        res = dc.mt.query_set(
            features=foreground,
            net=df_input,
            n_bg=n_bg,
            tmin=5,
            verbose=False,
        ).drop(columns=['pval']).rename(columns={'stat':'act'}).sort_values(by='padj', ascending=True)

        ## harmonize output
        res = res.copy()
        res["comparison"] = comparison
        res["gene_set"] = pk
        res["method"] = "ORA"
        res["direction"] = "up"
        res["n_foreground"] = len(foreground)
        res["n_background"] = n_bg

        res_lst.append(res)

df_enrich = pd.concat([df_enrich]+res_lst, ignore_index=True)

In [9]:
## Test overlap with query_set enrichment for unweighted gene sets 
## test for significantly DOWN-regulated DEGs

deg2 = deg.copy()
## keep valid DEA rows
deg2 = deg2.dropna(subset=[colname_comparison, colname_features, "padj", "stat"])
## make sure p-values/statistics are numeric
deg2["padj"] = pd.to_numeric(deg2["padj"], errors="coerce")
deg2["stat"] = pd.to_numeric(deg2["stat"], errors="coerce")
deg2 = deg2.dropna(subset=["padj", "stat"])

## run Fisher's
input_lst = [
    ("go_bp", df_go_bp),
    ("go_mf", df_go_mf),
    ("go_cc", df_go_cc),
    ("hallmark", df_hallmark),
    ("kegg", df_kegg),
    ("reactome", df_reactome),
]
res_lst = []
for comparison, df_cmp in deg2.groupby(colname_comparison):
    ## background: all genes tested in this DEA contrast
    background_genes = set(df_cmp[colname_features].dropna().unique())
    n_bg = len(background_genes)
    ## foreground (e.g., significantly upregulated genes)
    foreground = set( df_cmp.loc[(df_cmp["padj"] < 0.05) & (df_cmp["stat"] < 0),colname_features,].dropna().unique() )
    ## skip if no foreground 
    if len(foreground) == 0:
        continue

    for pk, df_input in input_lst:
        ## run Fisher's
        res = dc.mt.query_set(
            features=foreground,
            net=df_input,
            n_bg=n_bg,
            tmin=5,
            verbose=False,
        ).drop(columns=['pval']).rename(columns={'stat':'act'}).sort_values(by='padj', ascending=True)

        ## harmonize output
        res = res.copy()
        res["comparison"] = comparison
        res["gene_set"] = pk
        res["method"] = "ORA"
        res["direction"] = "down"
        res["n_foreground"] = len(foreground)
        res["n_background"] = n_bg

        res_lst.append(res)

df_enrich = pd.concat([df_enrich]+res_lst, ignore_index=True)

df_enrich.to_csv(out_dea_enrich, sep="\t", index=False)

df_enrich.head()

,comparison,source,act,padj,gene_set,method,direction,n_foreground,n_background
0,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,SPI1,-8.541232,1.021975e-14,collectri,ulm,None,None,None
1,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,ETS1,-6.066729,4.747640e-07,collectri,ulm,None,None,None
2,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,RUNX1,-5.904523,8.557855e-07,collectri,ulm,None,None,None
3,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,NFATC1,-4.235497,4.085288e-03,collectri,ulm,None,None,None
4,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,IRF8,-3.671292,3.451359e-02,collectri,ulm,None,None,None


#### **plot: CollecTRI**

In [10]:
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[plotTab['gene_set']=='collectri']
## select top TFs for each cluster
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']=[-np.log10(p) * 1 if act>1 else -np.log10(p) * -1 for act,p in zip(plotTab['act'],plotTab['padj'])]
## order by activity
plotTab=plotTab.sort_values(by='act',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

plotTab

,comparison,source,act,padj,gene_set,method,direction,n_foreground,n_background,sig
0,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,SPI1,-8.541232,1.021975e-14,collectri,ulm,None,None,None,-13.990560
1,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,ETS1,-6.066729,4.747640e-07,collectri,ulm,None,None,None,-6.323522
2,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,RUNX1,-5.904523,8.557855e-07,collectri,ulm,None,None,None,-6.067635
3,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,NFATC1,-4.235497,4.085288e-03,collectri,ulm,None,None,None,-2.388777
4,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,IRF8,-3.671292,3.451359e-02,collectri,ulm,None,None,None,-1.462010
7,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,LMO2,-3.442306,5.636783e-02,collectri,ulm,None,None,None,-1.248969
6,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,SSRP1,-3.417906,5.636783e-02,collectri,ulm,None,None,None,-1.248969
9,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,IKZF1,-3.143599,1.191832e-01,collectri,ulm,None,None,None,-0.923785
8,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,REST,3.165694,1.191832e-01,collectri,ulm,None,None,None,0.923785
5,HR_case__IFNG_pos_vs_HR_control__IFNG_pos,SREBF1,3.570866,4.239234e-02,collectri,ulm,None,None,None,1.372713


In [11]:
# barplot


h,w=3,6

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='act'),color='black', stat='identity', show_legend=True,alpha=0.7) +
    p9.scale_fill_gradient2(low="#2166AC",mid="white",high="#B2182B",midpoint=0) +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.geom_hline(yintercept=np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_classic() +
    p9.labs(
        title="Top collecTRI TFs",
        subtitle='contrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj) * direction',
        fill='TF\nactivity'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        axis_line_y=p9.element_blank(), axis_ticks_major_y=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / "collectri.png", width=w, height=h, limitsize=False)
p.save(out_plots / "collectri.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/collectri.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/collectri.pdf


#### **plot: Progeny**

In [12]:
# barplot

pk='progeny'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[plotTab['gene_set']==pk]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']=[-np.log10(p) * 1 if act>1 else -np.log10(p) * -1 for act,p in zip(plotTab['act'],plotTab['padj'])]
## order by activity
plotTab=plotTab.sort_values(by='act',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,6

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='act'),color='black', stat='identity', show_legend=True,alpha=0.7) +
    p9.scale_fill_gradient2(low="#2166AC",mid="white",high="#B2182B",midpoint=0) +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.geom_hline(yintercept=np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_classic() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle='contrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj) * direction',
        fill='activity'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        axis_line_y=p9.element_blank(), axis_ticks_major_y=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/progeny.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/progeny.pdf


#### **plot: Hallmark**

In [ ]:
# barplot, up

pk='hallmark'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/hallmark__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/hallmark__up.pdf


In [33]:
# barplot, down

pk='hallmark'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/hallmark__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/hallmark__down.pdf


#### **plot: KEGG**

In [ ]:
# barplot, up

pk='kegg'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/kegg__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/kegg__up.pdf


In [ ]:
# barplot, down

pk='kegg'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/kegg__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/kegg__down.pdf


#### **plot: Reactome**

In [ ]:
# barplot, up

pk='reactome'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/reactome__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/reactome__up.pdf


In [ ]:
# barplot, down

pk='reactome'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/reactome__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/reactome__down.pdf


#### **plot: GO BP**

In [ ]:
# barplot, up

pk='go_bp'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_bp__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_bp__up.pdf


In [ ]:
# barplot, down

pk='go_bp'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_bp__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_bp__down.pdf


#### **plot: GO CC**

In [ ]:
# barplot, up

pk='go_cc'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_cc__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_cc__up.pdf


In [ ]:
# barplot, down

pk='go_cc'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_cc__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_cc__down.pdf


#### **plot: GO MF**

In [ ]:
# barplot, up

pk='go_mf'
direction='up'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_mf__up.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_mf__up.pdf


In [ ]:
# barplot, down

pk='go_mf'
direction='down'
# prepare data ---------------------
plotTab=df_enrich.copy()
plotTab=plotTab[(plotTab['gene_set']==pk)&(plotTab['direction']==direction)]
## select top
topn=10
plotTab=plotTab.head(topn)
## significance
plotTab['sig']= -np.log10(plotTab['padj']) 
## shorten name
maxlen=70
plotTab['source']=[x[:maxlen]+'...' if len(x)>maxlen else x for x in plotTab['source']]
## order by activity
plotTab=plotTab.sort_values(by='sig',ascending=True)
plotTab['source']=plotTab['source'].astype('category').cat.reorder_categories(plotTab['source'].unique(),ordered=True)

# make plot ---------------------
h,w=3,8

p=(
    p9.ggplot(plotTab, p9.aes(x='source', y='sig')) +
    p9.geom_col(p9.aes(fill='sig'),color='black', stat='identity', show_legend=True,alpha=1) +
    p9.scale_fill_cmap(cmap_name="viridis") +
    p9.geom_hline(yintercept=0, color="black", size=0.5) +
    p9.geom_hline(yintercept=-np.log10(0.05), color="black", size=0.5, linetype="--",alpha=0.7) +
    p9.coord_flip() +
    p9.theme_bw() +
    p9.labs(
        title=f"Top {topn} {pk}",
        subtitle=f'DEG direction: {direction}\ncontrast: IFNG+ HRcases vs IFNG+ HRcontrols\ndashed lines: padj 0.05',
        y='-log10(padj)',
        fill='-log10(padj)'
    ) +
    p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=12), 
        plot_subtitle=p9.element_text(hjust=0, size=7), 
        axis_title=p9.element_text(size=10), 
        axis_text=p9.element_text(size=8), 
        panel_border=p9.element_rect(color="black", fill=None, size=1.2),
        panel_grid_major=p9.element_blank(),
        panel_grid_minor=p9.element_blank(),
        legend_key_size=10,
        legend_text=p9.element_text(size=8),
        figure_size=(w, h),
    ) 
)
p.save(out_plots / f"{pk}__{direction}.png", width=w, height=h, limitsize=False)
p.save(out_plots / f"{pk}__{direction}.pdf", width=w, height=h, limitsize=False)
plt.close()

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_mf__down.png
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 3 in image.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /gpfs/lsdf02/sd22b002/projects/olabisi/output/DEA/plots/funcomics/go_mf__down.pdf
